In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

from lib import *

path = "../data/Fashion-MNIST"
X_train, y_train, X_test, y_test = load_MINST_dataset(path)
X_train, X_test = flatten_images(X_train, X_test)

FileNotFoundError: [Errno 2] No such file or directory: '../data/Fashion-MNIST/train-images-idx3-ubyte'

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd

def create_pipeline(model):
    return Pipeline([
        ('standardization', StandardScaler()),
        ('reduce_dim', PCA(n_components=200, random_state=42)),
        ('classifier', model)
    ], verbose=False)

def predict(models, X_test):
    combined_prediction = []
    for m in models:
        combined_prediction.append(m.predict(X_test))
    df = pd.DataFrame(combined_prediction)
    return df.mode().iloc[0].values

def predict_and_evaluate(models, X_test, y_test):
    y_pred = predict(models, X_test)
    return (y_pred == y_test).mean()

def split_data(num, X_train, y_train):
    total_size = X_train.shape[0]
    X_subset = []
    y_subset = []
    for i in range(num):
        X_subset.append(X_train[i * total_size//num: (i+1) * total_size//num ,])
        y_subset.append(y_train[i * total_size//num: (i+1) * total_size//num ,])
    return X_subset, y_subset

In [10]:
num_models = 9
X_subset, y_subset = split_data(9, X_train, y_train)

ensembled_models_mnist = [
    create_pipeline(SVC(kernel='linear', C = 0.1, random_state=7)),
    create_pipeline(SVC(kernel='linear', C = 0.1, random_state=7)),
    create_pipeline(SVC(kernel='linear', C = 0.1, random_state=7)),

    create_pipeline(SVC(kernel='rbf', C = 100, gamma=0.001, random_state=7)),
    create_pipeline(SVC(kernel='rbf', C = 100, gamma=0.001, random_state=7)),
    create_pipeline(SVC(kernel='rbf', C = 100, gamma=0.001, random_state=7)),

    create_pipeline(SVC(kernel='poly', C = 100, gamma=0.0005, degree = 2, random_state=7)),
    create_pipeline(SVC(kernel='poly', C = 100, gamma=0.0005, degree = 2, random_state=7)),
    create_pipeline(SVC(kernel='poly', C = 100, gamma=0.0005, degree = 2, random_state=7))
]
# Train all the models
start = time.perf_counter()
for i, model in enumerate(ensembled_models_mnist):
    model.fit(X_subset[i], y_subset[i])
training_time = time.perf_counter() - start
print(f"Training time: {training_time} s")
accuracy = predict_and_evaluate(ensembled_models_mnist, X_test, y_test)
print(f'Accuracy: {accuracy}, Error: {1 - accuracy}')

Training time: 17.732809299996006 s
Accuracy: 0.8745, Error: 0.12549999999999994
